# Phase 1 — inspect one actual traffic video

**REQUIREMENT:** inspect input evidence only. This notebook does not load a detector, tracker, ReID model, or count vehicles. It does not stabilize or enhance frames. No ground-truth dataset exists; this is not a benchmark.

**FACT — execution context supplied by the user:** the dataset is mounted in Kaggle at `/kaggle/input/datasets/chrisbiran/traffic-tracker-videos`. GitHub stores notebook source, not videos. The current Kaggle editor is <https://www.kaggle.com/code/chrisbiran/new-traffic-counter/edit>.

Run through **Dataset discovery**, copy one **actually discovered** full path into `VIDEO_PATH`, then restart the kernel and run all cells. The initial `None` intentionally stops before inspecting any video. Review native PNGs and temporal previews, fill the manual-review cell with evidence references from this run, and rerun the review/report cells. For another video, change `VIDEO_PATH` and restart/run all again; each inspection creates a unique artifact directory.

**PROPOSAL — inspection policy:** five representative positions by *sequentially decoded frame index*, plus short consecutive-frame bursts at those positions. Two decode passes avoid relying on inaccurate random seeks or reported frame counts. This can take several minutes; it keeps video frames out of memory except the current frame. A backend may stop at EOF or at damage; the final sampled frame is the **last decodable frame**, not proof of a complete file. Variable-frame-rate videos are sampled by index, not exact elapsed-time quartiles.

No installation or dataset download occurs in this notebook. Kaggle normally supplies the inspection dependencies; missing dependencies are reported explicitly. GPU visibility is recorded but a GPU is not required for Phase 1.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import io
import json
import math
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from uuid import uuid4

DATASET_ROOT = Path('/kaggle/input/datasets/chrisbiran/traffic-tracker-videos')
VIDEO_PATH = None  # Set to one full path printed by discovery. No automatic selection.
OUTPUT_ROOT = Path('/kaggle/working/video_inspection')
VIDEO_EXTENSIONS = ('.mp4', '.mov', '.avi', '.mkv', '.m4v', '.webm', '.mpg', '.mpeg', '.mts', '.m2ts')
SAMPLE_POINTS = (('beginning', 0.0), ('quarter', 0.25), ('middle', 0.5), ('three_quarters', 0.75), ('end', 1.0))

# PROPOSAL: a 2-second target, capped at 30 consecutive frames per burst.
# The cap may shorten the actual interval. These are evidence budgets, not shake thresholds.
BURST_SECONDS = 2.0
BURST_MAX_FRAMES = 30
UNKNOWN_FPS_BURST_FRAMES = 30  # Explicit frame-budget fallback; duration remains unknown.
PREVIEW_WIDTH = 480  # Display copies only; native decoded PNGs are retained unchanged.
UNKNOWN_FPS_PREVIEW_RATE = 10.0  # Artificial playback when source FPS is unknown.
HASH_VIDEO = True  # Streaming SHA-256; adds a read of the input without copying it.
FFPROBE_TIMEOUT_SECONDS = 30


In [ ]:
ENVIRONMENT = {
    'python_version': sys.version,
    'platform': platform.platform(),
    'logical_cpu_count': os.cpu_count(),
    'process_available_cpu_count': len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else None,
    'dependencies': {},
    'ffprobe_path': shutil.which('ffprobe'),
    'ffmpeg_path': shutil.which('ffmpeg'),
}
missing = []
for module_name in ('cv2', 'numpy', 'PIL', 'IPython'):
    try:
        module = importlib.import_module(module_name)
        ENVIRONMENT['dependencies'][module_name] = {'status': 'available', 'version': getattr(module, '__version__', None)}
    except ImportError as exc:
        ENVIRONMENT['dependencies'][module_name] = {'status': 'unavailable', 'error': str(exc)}
        missing.append(module_name)

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    try:
        gpu_result = subprocess.run(
            [nvidia_smi, '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=15, check=False,
        )
        ENVIRONMENT['gpu_probe'] = {
            'status': 'reported' if gpu_result.returncode == 0 else 'unknown_probe_failed',
            'returncode': gpu_result.returncode,
            'reported_csv': gpu_result.stdout.strip(),
            'error': gpu_result.stderr.strip() or None,
        }
    except (OSError, subprocess.TimeoutExpired) as exc:
        ENVIRONMENT['gpu_probe'] = {'status': 'unknown_probe_failed', 'error': str(exc)}
else:
    ENVIRONMENT['gpu_probe'] = {'status': 'unknown_nvidia_smi_unavailable'}

for tool_name in ('ffprobe', 'ffmpeg'):
    executable = ENVIRONMENT[f'{tool_name}_path']
    if executable:
        try:
            result = subprocess.run([executable, '-version'], capture_output=True, text=True, timeout=15, check=False)
            ENVIRONMENT[f'{tool_name}_version'] = result.stdout.splitlines()[0] if result.returncode == 0 and result.stdout else None
        except (OSError, subprocess.TimeoutExpired):
            ENVIRONMENT[f'{tool_name}_version'] = None

print(json.dumps(ENVIRONMENT, indent=2))
print('GPU probe failure/unavailability does not establish absence of a GPU or CUDA compatibility.')
if missing:
    raise RuntimeError(f'Missing inspection dependencies: {missing}. Configure them in the runtime, then restart and rerun. No automatic installation was performed.')

import cv2
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import HTML, Markdown, display


In [ ]:
def positive_number(value, integer=False):
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    if not math.isfinite(number) or number <= 0:
        return None
    if integer:
        return int(number) if number.is_integer() else None
    return number


def rational_rate(value):
    try:
        numerator, denominator = str(value).split('/')
        return positive_number(float(numerator) / float(denominator))
    except (ValueError, ZeroDivisionError):
        return None


def inspect_metadata(path):
    path = Path(path)
    record = {
        'path': str(path), 'filename': path.name, 'extension': path.suffix,
        'size_bytes': path.stat().st_size, 'opencv': {}, 'ffprobe': {}, 'warnings': [],
    }
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            record['opencv'] = {'status': 'unavailable_cannot_open'}
        else:
            fps = positive_number(cap.get(cv2.CAP_PROP_FPS))
            frame_count = positive_number(cap.get(cv2.CAP_PROP_FRAME_COUNT), integer=True)
            fourcc_value = positive_number(cap.get(cv2.CAP_PROP_FOURCC), integer=True)
            fourcc = ''.join(chr((fourcc_value >> (8 * i)) & 255) for i in range(4)) if fourcc_value else None
            if fourcc and any(not char.isprintable() for char in fourcc):
                fourcc = None
            record['opencv'] = {
                'status': 'reported', 'backend': cap.getBackendName(),
                'width': positive_number(cap.get(cv2.CAP_PROP_FRAME_WIDTH), integer=True),
                'height': positive_number(cap.get(cv2.CAP_PROP_FRAME_HEIGHT), integer=True),
                'fps': fps, 'frame_count': frame_count, 'codec_fourcc': fourcc,
                'duration_seconds_derived': frame_count / fps if frame_count and fps else None,
                'provenance': 'OpenCV backend-reported properties; frame_count / fps is an estimate, not a measured duration',
            }
    except (cv2.error, OSError, ValueError) as exc:
        record['opencv'] = {'status': 'probe_failed', 'error': str(exc)}
    finally:
        cap.release()

    ffprobe = shutil.which('ffprobe')
    if not ffprobe:
        record['ffprobe'] = {'status': 'unavailable_executable_missing'}
        record['warnings'].append('Explicit fallback: only OpenCV backend metadata is available; derived duration is an estimate.')
        return record
    try:
        result = subprocess.run(
            [ffprobe, '-v', 'error', '-select_streams', 'v:0', '-show_entries',
             'stream=index,codec_name,width,height,r_frame_rate,avg_frame_rate,nb_frames,duration:format=duration,format_name',
             '-of', 'json', str(path)],
            capture_output=True, text=True, timeout=FFPROBE_TIMEOUT_SECONDS, check=False,
        )
        if result.returncode != 0:
            record['ffprobe'] = {'status': 'probe_failed', 'error': result.stderr.strip(), 'returncode': result.returncode}
        else:
            raw = json.loads(result.stdout)
            streams = raw.get('streams', [])
            stream = streams[0] if streams else {}
            container = raw.get('format', {})
            record['ffprobe'] = {
                'status': 'reported' if streams else 'no_video_stream',
                'width': positive_number(stream.get('width'), integer=True),
                'height': positive_number(stream.get('height'), integer=True),
                'codec_name': stream.get('codec_name'),
                'frame_count': positive_number(stream.get('nb_frames'), integer=True),
                'average_fps': rational_rate(stream.get('avg_frame_rate')),
                'nominal_fps': rational_rate(stream.get('r_frame_rate')),
                'average_fps_raw': stream.get('avg_frame_rate'),
                'nominal_fps_raw': stream.get('r_frame_rate'),
                'stream_duration_seconds': positive_number(stream.get('duration')),
                'container_duration_seconds': positive_number(container.get('duration')),
                'container_format': container.get('format_name'),
                'provenance': 'ffprobe first video stream and container metadata; container duration may include other streams',
                'stderr': result.stderr.strip() or None,
            }
    except (OSError, subprocess.TimeoutExpired, json.JSONDecodeError) as exc:
        record['ffprobe'] = {'status': 'probe_failed', 'error': str(exc)}
    if record['ffprobe']['status'] != 'reported':
        record['warnings'].append('Explicit fallback: ffprobe metadata unavailable; inspect the failure and OpenCV estimates.')

    cv_meta, ff_meta = record['opencv'], record['ffprobe']
    for key in ('width', 'height', 'frame_count'):
        if cv_meta.get(key) is not None and ff_meta.get(key) is not None and cv_meta[key] != ff_meta[key]:
            record['warnings'].append(f'Backend discrepancy for {key}: OpenCV={cv_meta[key]}, ffprobe={ff_meta[key]}.')
    avg, nominal = ff_meta.get('average_fps'), ff_meta.get('nominal_fps')
    if avg and nominal and avg != nominal:
        record['warnings'].append('Average and nominal frame rates differ. This alone does not prove variable frame rate; inspect timestamps if needed.')
    if avg and cv_meta.get('fps') and avg != cv_meta['fps']:
        record['warnings'].append(f'FPS discrepancy: OpenCV={cv_meta["fps"]}, ffprobe average={avg}. Semantics differ; neither proves constant FPS.')
    return record


def discover_videos(root, extensions):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'Dataset mount unavailable: {root}. Attach the dataset and execute in Kaggle. No video was inspected.')
    extensions = {str(ext).lower() for ext in extensions}
    paths = sorted((path.resolve() for path in root.rglob('*') if path.is_file() and path.suffix.lower() in extensions), key=str)
    records = []
    for path in paths:
        try:
            records.append(inspect_metadata(path))
        except OSError as exc:
            records.append({'path': str(path), 'filename': path.name, 'extension': path.suffix, 'size_bytes': None, 'opencv': {}, 'ffprobe': {}, 'warnings': [str(exc)]})
    return paths, records


def metadata_summary(record):
    cv_meta, ff_meta = record.get('opencv', {}), record.get('ffprobe', {})
    def choose(key):
        if ff_meta.get(key) is not None:
            return {'value': ff_meta[key], 'source': f'ffprobe.{key}'}
        return {'value': cv_meta.get(key), 'source': f'OpenCV.{key}' if cv_meta.get(key) is not None else 'unknown'}
    summary = {key: choose(key) for key in ('width', 'height', 'frame_count')}
    if ff_meta.get('average_fps'):
        summary['fps'] = {'value': ff_meta['average_fps'], 'source': 'ffprobe average frame rate; constant-FPS status unknown'}
    else:
        summary['fps'] = {'value': cv_meta.get('fps'), 'source': 'OpenCV backend FPS' if cv_meta.get('fps') else 'unknown'}
    duration_key = next((key for key in ('stream_duration_seconds', 'container_duration_seconds') if ff_meta.get(key) is not None), None)
    if duration_key:
        summary['duration_seconds'] = {'value': ff_meta[duration_key], 'source': f'ffprobe.{duration_key}'}
    else:
        derived = cv_meta.get('duration_seconds_derived')
        summary['duration_seconds'] = {'value': derived, 'source': 'OpenCV frame_count / fps estimate' if derived else 'unknown'}
    summary['codec'] = {'value': ff_meta.get('codec_name') or cv_meta.get('codec_fourcc'), 'source': 'ffprobe codec_name' if ff_meta.get('codec_name') else ('OpenCV FOURCC' if cv_meta.get('codec_fourcc') else 'unknown')}
    return summary


## Dataset discovery — metadata only

Candidate suffixes are configurable; extensions alone do not establish that a file is decodable. Every discovered candidate is probed for basic metadata, but only the explicitly selected video receives full decode/frame sampling. No weekday/weekend, time of day, store identity, or filename convention is inferred. Missing values remain `null`/unknown. ffprobe is optional: failures and the OpenCV fallback are recorded.


In [ ]:
DISCOVERED_PATHS, DISCOVERY_RECORDS = discover_videos(DATASET_ROOT, VIDEO_EXTENSIONS)
print(f'Discovered {len(DISCOVERED_PATHS)} video candidates under {DATASET_ROOT}:')
for record in DISCOVERY_RECORDS:
    print(json.dumps({'path': record['path'], 'filename': record['filename'], 'extension': record['extension'],
                      'size_bytes': record['size_bytes'], 'metadata': metadata_summary(record),
                      'opencv_status': record.get('opencv', {}).get('status', 'unknown'),
                      'ffprobe_status': record.get('ffprobe', {}).get('status', 'unknown'),
                      'warnings': record['warnings']}, indent=2))
if not DISCOVERED_PATHS:
    raise RuntimeError('No candidate video files discovered. Check the Kaggle mount and VIDEO_EXTENSIONS; no video was inspected.')


In [ ]:
if VIDEO_PATH is None:
    raise RuntimeError('Set VIDEO_PATH to ONE full path printed above, then restart the kernel and run all cells. No video is selected automatically.')
SELECTED_VIDEO = Path(VIDEO_PATH).expanduser().resolve()
if SELECTED_VIDEO not in DISCOVERED_PATHS or not SELECTED_VIDEO.is_file():
    raise ValueError(f'VIDEO_PATH must be one existing discovered candidate: {SELECTED_VIDEO}')
SELECTED_METADATA = next(record for record in DISCOVERY_RECORDS if record['path'] == str(SELECTED_VIDEO))
SELECTED_SUMMARY = metadata_summary(SELECTED_METADATA)
print(json.dumps({'selected_video': str(SELECTED_VIDEO), 'metadata': SELECTED_SUMMARY,
                  'all_backend_evidence': SELECTED_METADATA}, indent=2))


In [ ]:
def count_decoded_frames(path):
    cap = cv2.VideoCapture(str(path))
    count, first_dimensions = 0, None
    try:
        if not cap.isOpened():
            raise RuntimeError(f'OpenCV cannot open selected video: {path}')
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if first_dimensions is None:
                first_dimensions = {'width': int(frame.shape[1]), 'height': int(frame.shape[0])}
            count += 1
            if count % 1500 == 0:
                print(f'First pass: {count} frames decoded...', flush=True)
    finally:
        cap.release()
    if count == 0:
        raise RuntimeError('No frames decoded from selected video. EOF/corruption/unsupported codec must be investigated.')
    return {'decoded_frame_count': count, 'first_decoded_dimensions': first_dimensions,
            'stop_reason': 'read_returned_false: EOF_or_decode_failure_not_distinguished',
            'completeness': 'not_guaranteed_by_OpenCV'}


def streaming_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def sample_video(path, run_dir, decoded_count, fps):
    run_dir = Path(run_dir)
    (run_dir / 'frames').mkdir(parents=True, exist_ok=True)
    if BURST_SECONDS <= 0 or BURST_MAX_FRAMES < 1 or UNKNOWN_FPS_BURST_FRAMES < 1 or PREVIEW_WIDTH < 1:
        raise ValueError('Inspection budgets and preview width must be positive.')
    burst_frames = min(BURST_MAX_FRAMES, max(1, int(round(BURST_SECONDS * fps)))) if fps else min(BURST_MAX_FRAMES, UNKNOWN_FPS_BURST_FRAMES)
    representatives, burst_plans = [], []
    for label, fraction in SAMPLE_POINTS:
        if not 0 <= fraction <= 1:
            raise ValueError('Representative fractions must lie in [0, 1].')
        index = int(round((decoded_count - 1) * fraction))
        representatives.append({'label': label, 'requested_fraction_of_decoded_index_range': fraction, 'requested_index': index})
        start = max(0, min(index - burst_frames // 2, decoded_count - burst_frames))
        indices = list(range(start, min(decoded_count, start + burst_frames)))
        burst_plans.append({'label': label, 'requested_anchor_index': index, 'requested_indices': indices})
    wanted = {item['requested_index'] for item in representatives}
    for burst in burst_plans:
        wanted.update(burst['requested_indices'])
    evidence = {}
    cap = cv2.VideoCapture(str(path))
    second_count = 0
    try:
        if not cap.isOpened():
            raise RuntimeError('Second-pass decoder cannot open video.')
        while second_count <= max(wanted):
            ok, frame = cap.read()
            if not ok:
                break
            index = second_count
            second_count += 1
            if index not in wanted:
                continue
            path_relative = f'frames/frame_{index:09d}.png'
            if not cv2.imwrite(str(run_dir / path_relative), frame):
                raise OSError(f'Could not save decoded frame {index}.')
            backend_msec = cap.get(cv2.CAP_PROP_POS_MSEC)
            backend_seconds = backend_msec / 1000 if math.isfinite(backend_msec) and backend_msec >= 0 else None
            evidence[index] = {
                'sequential_decoded_index_zero_based': index, 'path': path_relative,
                'width': int(frame.shape[1]), 'height': int(frame.shape[0]),
                'opencv_reported_timestamp_seconds': backend_seconds,
                'opencv_timestamp_provenance': 'CAP_PROP_POS_MSEC after sequential read; backend dependent, may be unavailable/repeated/inaccurate, not verified source PTS',
                'estimated_timestamp_seconds_from_fps': index / fps if fps else None,
                'estimated_timestamp_provenance': 'decoded index / configured metadata FPS; approximate only, unreliable for VFR or decode gaps' if fps else 'unknown FPS: no derived timestamp',
                'pixel_provenance': 'native OpenCV decoded BGR pixels saved losslessly as PNG; decoder may apply orientation/color conversion, no additional enhancement or geometry overlay',
            }
    finally:
        cap.release()
    for item in representatives:
        item['evidence'] = evidence.get(item['requested_index'])
        item['status'] = 'saved' if item['evidence'] else 'missing_second_pass_decode'
    bursts = []
    for plan in burst_plans:
        folder = run_dir / 'bursts' / plan['label']
        folder.mkdir(parents=True, exist_ok=True)
        observed = [evidence[index] for index in plan['requested_indices'] if index in evidence]
        data = {**plan, 'frames': observed, 'status': 'saved' if len(observed) == len(plan['requested_indices']) else 'partial_decode',
                'fps_basis': fps, 'estimated_first_to_last_seconds': (len(observed) - 1) / fps if fps and observed else None,
                'rate_provenance': 'metadata-FPS approximation; not verified real-time playback' if fps else 'unknown source FPS; frame-budget burst has no derived real duration',
                'manifest_path': f'bursts/{plan["label"]}/manifest.json'}
        (folder / 'manifest.json').write_text(json.dumps(data, indent=2), encoding='utf-8')
        bursts.append(data)
    return {'representatives': representatives, 'bursts': bursts, 'frame_evidence': list(evidence.values()),
            'missing_requested_indices': sorted(wanted - set(evidence)), 'second_pass_decoded_count': second_count,
            'sampling_basis': 'fractions of first-pass sequential decoded indices; last sample is last first-pass decodable frame, not proven full-video end'}


def make_contact_sheet(sampling, run_dir):
    items = [item for item in sampling['representatives'] if item['evidence']]
    if not items:
        return None
    width, height = 480, 320
    sheet = Image.new('RGB', (width * min(3, len(items)), height * math.ceil(len(items) / 3)), '#202020')
    draw = ImageDraw.Draw(sheet)
    for position, item in enumerate(items):
        entry = item['evidence']
        with Image.open(Path(run_dir) / entry['path']) as raw:
            thumbnail = raw.convert('RGB')
        thumbnail.thumbnail((width, height - 48))
        x, y = (position % 3) * width, (position // 3) * height
        sheet.paste(thumbnail, (x, y))
        draw.text((x + 5, y + height - 42), f'{item["label"]}: index {entry["sequential_decoded_index_zero_based"]}', fill='white')
        draw.text((x + 5, y + height - 24), f'Native {entry["width"]} x {entry["height"]}; display thumbnail', fill='white')
    sheet.save(Path(run_dir) / 'contact_sheet.png')
    return 'contact_sheet.png'


def make_temporal_preview(burst, run_dir):
    # Small display copies only. Native PNG evidence remains unchanged on disk.
    import base64
    entries = []
    for frame in burst['frames']:
        with Image.open(Path(run_dir) / frame['path']) as raw:
            thumbnail = raw.convert('RGB')
        thumbnail.thumbnail((PREVIEW_WIDTH, PREVIEW_WIDTH))
        buffer = io.BytesIO()
        thumbnail.save(buffer, format='PNG')
        entries.append({'image': 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii'),
                        'index': frame['sequential_decoded_index_zero_based'], 'raw': frame['path']})
    if not entries:
        return None
    rate = burst['fps_basis'] or UNKNOWN_FPS_PREVIEW_RATE
    description = 'Approximate metadata-FPS playback; source timing not verified.' if burst['fps_basis'] else f'Artificial {UNKNOWN_FPS_PREVIEW_RATE} FPS playback; source FPS unknown.'
    html = """<!doctype html><meta charset="utf-8"><title>Temporal inspection</title>
<p>Unenhanced consecutive-frame preview; resized for display only. Native PNGs are retained.</p>
<p id="timing"></p><button id="toggle">Play / pause</button>
<input id="position" type="range" min="0" value="0"><p id="label"></p><img id="frame" alt="Decoded frame preview">
<script>
const frames=ENTRIES, rate=RATE;
let position=0, timer=null;
const slider=document.getElementById('position');slider.max=frames.length-1;
function show(){const frame=frames[position];document.getElementById('frame').src=frame.image;
 document.getElementById('label').textContent='Sequential index '+frame.index+'; native evidence '+frame.raw;slider.value=position;}
slider.oninput=()=>{position=Number(slider.value);show();};
document.getElementById('toggle').onclick=()=>{if(timer){clearInterval(timer);timer=null;}else{timer=setInterval(()=>{position=(position+1)%frames.length;show();},1000/rate);}};
document.getElementById('timing').textContent=DESCRIPTION;show();
</script>"""
    html = html.replace('DESCRIPTION', json.dumps(description)).replace('RATE', json.dumps(rate)).replace('ENTRIES', json.dumps(entries))
    relative = f'bursts/{burst["label"]}/preview.html'
    (Path(run_dir) / relative).write_text(html, encoding='utf-8')
    return relative


## Decode and retain evidence

Native decoded frames are saved once even if short-video sample positions overlap. All five labels remain in the manifest. Raw videos are never copied. Native dimensions and both backend-reported and FPS-derived time references are retained; neither timestamp source is assumed exact. Contact-sheet/preview resizing is explicitly display-only. Use native PNGs for pixel-scale visibility and geometry review.

Short bursts are local evidence, not a survey of every moment. For shake assessment, compare static background/road features over consecutive frames rather than infer camera movement from moving vehicles. Manually view the original selected video in Kaggle as needed to inspect rare events or the complete recording; no model processing is involved. If the bursts are insufficient, increase the explicit budgets and run a fresh inspection.

If inline HTML previews are blocked by notebook trust/browser restrictions, download and open each saved `bursts/.../preview.html` locally, or inspect its consecutive native PNGs. The report keeps their evidence references.


In [ ]:
run_name = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT.expanduser().resolve() / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
RUN_DATA = {
    'run_id': run_name, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'started', 'environment': ENVIRONMENT,
    'configuration': {'dataset_root': str(DATASET_ROOT), 'video_path': str(SELECTED_VIDEO),
                      'output_root': str(OUTPUT_ROOT), 'extensions': list(VIDEO_EXTENSIONS),
                      'sample_points': list(SAMPLE_POINTS), 'burst_seconds_proposal': BURST_SECONDS,
                      'burst_max_frames_proposal': BURST_MAX_FRAMES, 'unknown_fps_burst_frames_proposal': UNKNOWN_FPS_BURST_FRAMES,
                      'preview_width': PREVIEW_WIDTH, 'unknown_fps_preview_rate_artificial': UNKNOWN_FPS_PREVIEW_RATE,
                      'hash_video': HASH_VIDEO, 'ffprobe_timeout_seconds': FFPROBE_TIMEOUT_SECONDS},
    'input_identity': {'path': str(SELECTED_VIDEO), 'filename': SELECTED_VIDEO.name,
                       'size_bytes': SELECTED_VIDEO.stat().st_size, 'sha256': None},
    'discovery': DISCOVERY_RECORDS, 'selected_metadata': SELECTED_METADATA, 'metadata_summary': SELECTED_SUMMARY,
    'warnings': list(SELECTED_METADATA['warnings']),
}
try:
    if HASH_VIDEO:
        RUN_DATA['input_identity']['sha256'] = streaming_sha256(SELECTED_VIDEO)
    RUN_DATA['decode'] = count_decoded_frames(SELECTED_VIDEO)
    decoded_count = RUN_DATA['decode']['decoded_frame_count']
    fps_basis = SELECTED_SUMMARY['fps']['value']
    RUN_DATA['sampling'] = sample_video(SELECTED_VIDEO, RUN_DIR, decoded_count, fps_basis)
    for backend in ('opencv', 'ffprobe'):
        reported = SELECTED_METADATA[backend].get('frame_count')
        if reported is not None and reported != decoded_count:
            RUN_DATA['warnings'].append(f'{backend} reported {reported} frames, first-pass OpenCV decoded {decoded_count}; investigate truncation/metadata/decoder discrepancy.')
    if RUN_DATA['sampling']['missing_requested_indices']:
        RUN_DATA['warnings'].append('Second pass failed to retain some requested evidence; do not treat the inspection as complete.')
    RUN_DATA['contact_sheet'] = make_contact_sheet(RUN_DATA['sampling'], RUN_DIR)
    for burst in RUN_DATA['sampling']['bursts']:
        burst['preview_path'] = make_temporal_preview(burst, RUN_DIR)
    RUN_DATA['status'] = 'evidence_saved_manual_review_pending' if not RUN_DATA['sampling']['missing_requested_indices'] else 'partial_evidence_manual_review_pending'
except Exception as exc:
    RUN_DATA['status'] = 'inspection_failed'
    RUN_DATA['failure'] = {'type': type(exc).__name__, 'message': str(exc)}
    (RUN_DIR / 'failure.json').write_text(json.dumps(RUN_DATA, indent=2), encoding='utf-8')
    print(f'Inspection failed. Failure evidence: {RUN_DIR / "failure.json"}')
    raise
finally:
    (RUN_DIR / 'run_manifest.json').write_text(json.dumps(RUN_DATA, indent=2), encoding='utf-8')

print(f'Artifacts: {RUN_DIR}')
print(json.dumps({'status': RUN_DATA['status'], 'decode': RUN_DATA['decode'],
                  'samples': RUN_DATA['sampling']['representatives'], 'warnings': RUN_DATA['warnings']}, indent=2))


In [ ]:
if RUN_DATA['contact_sheet']:
    display(Image.open(RUN_DIR / RUN_DATA['contact_sheet']))
for burst in RUN_DATA['sampling']['bursts']:
    print(f'Temporal evidence {burst["label"]}: {len(burst["frames"])} consecutive frames; manifest {burst["manifest_path"]}; preview {burst["preview_path"]}')
    if burst['preview_path']:
        # Each isolated iframe can play/pause or scrub without executing source HTML in the parent page.
        import base64
        encoded = base64.b64encode((RUN_DIR / burst['preview_path']).read_bytes()).decode('ascii')
        display(HTML(f'<iframe sandbox="allow-scripts" src="data:text/html;base64,{encoded}" width="700" height="650" title="Temporal preview {burst["label"]}"></iframe>'))


## Human visual review — unknown until inspected

**REQUIREMENT:** enter actual observations with references to this run's `frames/...png`, `bursts/.../manifest.json`, or `contact_sheet.png`. Keep an item `None` when evidence is insufficient. The notebook does not automatically classify lighting, shake, geometry, viewing angle, or vehicle behavior. An oblique road view does not establish a calibrated 45-degree angle.

For `camera_stability`, choose exactly one human qualitative category when supported: `effectively_stationary`, `mildly_shaky`, `moderately_shaky`, or `severely_unstable`; describe evidence and sampling limitations. A still frame/contact sheet alone cannot establish temporal shake. For geometry feasibility, discuss possible per-video Side A / counting line / Side B placement and alignment limitations without selecting coordinates, anchors, or implementation rules. The reference screenshot's side mapping does not automatically apply to this video.

For low light, record observed visibility separately from **HYPOTHESIS_UNTESTED** risks for detection, tracking, and ReID. No enhancement is performed; night preprocessing remains Phase 7. Potential downstream failures are hypotheses until the relevant pipeline stage is executed. Questions remain **OPEN**. Recording period/store labels are unknown unless supplied separately; do not infer them from filenames.


In [ ]:
# Edit only after inspecting evidence. Example shape: {'observation': 'actual note', 'evidence': ['frames/frame_000000000.png']}.
# Do not copy a note or evidence reference from a different run/video.
MANUAL_REVIEW = {
    'camera_position': {'observation': None, 'evidence': []},
    'camera_orientation': {'observation': None, 'evidence': []},
    'road_orientation': {'observation': None, 'evidence': []},
    'approximate_viewing_angle': {'observation': None, 'evidence': []},
    'handheld_shake': {'observation': None, 'evidence': []},
    'camera_stability': {'observation': None, 'category': None, 'evidence': []},
    'lighting_condition': {'observation': None, 'evidence': []},
    'vehicle_visibility': {'observation': None, 'evidence': []},
    'occlusion': {'observation': None, 'evidence': []},
    'blur': {'observation': None, 'evidence': []},
    'reflections': {'observation': None, 'evidence': []},
    'shadows': {'observation': None, 'evidence': []},
    'vehicles_entering_exiting': {'observation': None, 'evidence': []},
    'diagonal_vehicle_movement': {'observation': None, 'evidence': []},
    'geometry_feasibility': {'observation': None, 'evidence': []},
    'geometry_alignment_over_time': {'observation': None, 'evidence': []},
    'low_light_detection_risk': {'observation': None, 'evidence': [], 'claim_type': 'HYPOTHESIS_UNTESTED'},
    'low_light_tracking_risk': {'observation': None, 'evidence': [], 'claim_type': 'HYPOTHESIS_UNTESTED'},
    'low_light_reid_risk': {'observation': None, 'evidence': [], 'claim_type': 'HYPOTHESIS_UNTESTED'},
    'potential_failure_cases': {'observation': None, 'evidence': [], 'claim_type': 'HYPOTHESIS_UNTESTED'},
    'technical_questions': {'observation': None, 'evidence': [], 'claim_type': 'OPEN'},
}


In [ ]:
def write_report(run_data, manual_review, run_dir):
    run_dir = Path(run_dir)
    allowed_refs = {frame['path'] for frame in run_data['sampling']['frame_evidence']}
    allowed_refs.update(burst['manifest_path'] for burst in run_data['sampling']['bursts'])
    allowed_refs.update(burst['preview_path'] for burst in run_data['sampling']['bursts'] if burst['preview_path'])
    if run_data.get('contact_sheet'):
        allowed_refs.add(run_data['contact_sheet'])
    temporal_refs = set()
    for burst in run_data['sampling']['bursts']:
        indices = [frame['sequential_decoded_index_zero_based'] for frame in burst['frames']]
        if len(indices) >= 2 and all(right == left + 1 for left, right in zip(indices, indices[1:])):
            temporal_refs.add(burst['manifest_path'])
            if burst.get('preview_path'):
                temporal_refs.add(burst['preview_path'])
    shake_categories = {'effectively_stationary', 'mildly_shaky', 'moderately_shaky', 'severely_unstable'}
    for field, entry in manual_review.items():
        if field == 'camera_stability' and (entry.get('category') is not None or entry.get('observation') is not None):
            observation = entry.get('observation')
            if entry.get('category') not in shake_categories or not isinstance(observation, str) or not observation.strip():
                raise ValueError('Camera stability requires both one allowed qualitative category and an actual observation.')
        if entry.get('observation') is not None and field != 'technical_questions' and not entry.get('evidence'):
            raise ValueError(f'{field} requires an evidence reference from this run.')
        for reference in entry.get('evidence', []):
            if reference not in allowed_refs or not (run_dir / reference).is_file():
                raise ValueError(f'Unknown/missing evidence reference for {field}: {reference}')
        if field in ('handheld_shake', 'camera_stability', 'geometry_alignment_over_time') and entry.get('observation') is not None:
            if not any(reference in temporal_refs for reference in entry.get('evidence', [])):
                raise ValueError(f'{field} requires a burst with at least two retained consecutive frames; stills and one-frame bursts do not establish movement.')
    unknown = [field for field, entry in manual_review.items() if entry.get('observation') is None]
    report = {**run_data, 'manual_review': manual_review,
              'review_status': 'incomplete_unknown_items' if unknown else 'human_review_recorded_not_a_benchmark',
              'unknown_review_items': unknown,
              'limitations': [
                  'No detector/tracker/ReID/counting execution or ground-truth evaluation.',
                  'Human observations depend on inspected evidence; unknown items are not fabricated.',
                  'Representative fractions use decoded frame indices; VFR elapsed-time quartiles may differ.',
                  'OpenCV read failure does not distinguish EOF from damaged/unsupported frames.',
                  'Backend timestamps and FPS-derived timing are recorded with uncertainty.',
                  'Short bursts can miss rare shake/events; inspect more evidence if needed.',
                  'Display copies are resized; geometry/visibility assessment should also use native PNGs.',
              ]}
    (run_dir / 'inspection_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    def text(value):
        return 'UNKNOWN' if value is None else str(value).replace('|', '\\|').replace('\n', '<br>')
    lines = ['# Phase 1 — one-video inspection report', '', f'Run: `{run_data["run_id"]}`',
             f'Input: `{run_data["input_identity"]["path"]}`', f'Status: **{run_data["status"]}**; review: **{report["review_status"]}**',
             f'SHA-256: `{text(run_data["input_identity"]["sha256"])}`', '',
             '## Metadata (executed probe/decode evidence)', '', '| Field | Value | Provenance |', '|---|---|---|',
             f'| Filename | {text(run_data["input_identity"]["filename"])} | selected discovered file |',
             f'| File size bytes | {run_data["input_identity"]["size_bytes"]} | filesystem stat |']
    for field, item in run_data['metadata_summary'].items():
        lines.append(f'| {field} | {text(item["value"])} | {text(item["source"])} |')
    lines.append(f'| Decoded frame count | {run_data["decode"]["decoded_frame_count"]} | sequential OpenCV first pass; completeness unverified |')
    lines += ['', '## Manual visual observations', '', '| Item | Observation | Evidence |', '|---|---|---|']
    for field, entry in manual_review.items():
        observation = text(entry.get('observation'))
        if entry.get('category'):
            observation = text(entry['category']) + ': ' + observation
        claim = entry.get('claim_type', 'HUMAN_OBSERVATION' if entry.get('observation') is not None else 'UNKNOWN')
        lines.append(f'| {field} ({claim}) | {observation} | {text(", ".join(entry.get("evidence", [])) or None)} |')
    lines += ['', '## Decode warnings and limitations', '']
    lines += ['- ' + warning for warning in run_data['warnings'] + report['limitations']]
    lines += ['', '## Evidence', '', '- Native frames and burst manifests: see `run_manifest.json`.',
              '- The contact sheet and temporal previews are display aids; native PNGs retain decoded dimensions.',
              '- No Phase 2 work is started or validated by this report.', '']
    markdown = '\n'.join(lines)
    (run_dir / 'inspection_report.md').write_text(markdown, encoding='utf-8')
    return report, markdown

INSPECTION_REPORT, REPORT_MARKDOWN = write_report(RUN_DATA, MANUAL_REVIEW, RUN_DIR)
display(Markdown(REPORT_MARKDOWN))
print(f'Reports: {RUN_DIR / "inspection_report.md"} and {RUN_DIR / "inspection_report.json"}')


## Finish Phase 1

Download the run's `inspection_report.md`, `inspection_report.json`, and needed retained evidence from Kaggle outputs. Update repository findings only after reviewing this executed evidence; no actual-video findings are embedded in the output-free source notebook. Keep all videos out of GitHub. Unknown review items and decode discrepancies remain open. Stop here: Phase 2 requires a separate task and verified model/runtime/class-mapping decisions.
